# Real-Time Mode
## One line turns a micro-batch stream into sub-second streaming.

Order events flow through stateless **guardrails** (flag oversized orders, too many items, leaked credentials) and get routed **ALLOW** / **QUARANTINE** — live. The point of this demo: enabling Spark 4.1 **Real-Time Mode** is a **single-line trigger change**.

---
## The guardrails (stateless checks)
Plain column logic — no windows, no state, which is exactly what Real-Time Mode requires:

In [ ]:
from IPython.display import Code
Code(filename='/home/jovyan/demos/realtime-mode/rtm_stream_lib.py')

---
## The one-line difference
Real-Time Mode is **not** a new query — it's the same stream with a different **trigger**. There's no native PySpark kwarg for it yet, so we reach through the JVM:

```python
# Normal micro-batch trigger (what you'd write today):
writer.trigger(processingTime='5 seconds').start()

# Real-Time Mode — the ONLY change:
rt = spark._jvm.org.apache.spark.sql.streaming.Trigger.RealTime('5 seconds')
writer._jwrite.trigger(rt).start()
```
`start_query(spark, use_realtime=...)` flips exactly that one line.

In [ ]:
import sys, time
sys.path.insert(0, '/home/jovyan/demos/realtime-mode')
import rtm_stream_lib as rtm
from pyspark.sql import SparkSession
spark = SparkSession.builder.config('spark.sql.streaming.realTimeMode.allowlistCheck','false').getOrCreate()
rtm.start_producer(rows_per_sec=20)   # drip order events into Kafka
print('producing order events to Kafka...')

### Baseline: normal micro-batch trigger

In [ ]:
q = rtm.start_query(spark, use_realtime=False, query_name='mb')
time.sleep(12)
spark.sql('SELECT decision, count(*) AS n FROM mb GROUP BY decision').show()
spark.sql("SELECT * FROM mb WHERE decision='QUARANTINE'").show(5, truncate=False)
q.stop()

### Flip to Real-Time Mode — the one-line change
Same query, same guardrails. Only the trigger changed:

In [ ]:
q = rtm.start_query(spark, use_realtime=True, query_name='rt')   # <-- use_realtime=True
time.sleep(12)
spark.sql('SELECT decision, count(*) AS n FROM rt GROUP BY decision').show()
spark.sql("SELECT order_id, brand, total, reasons FROM rt WHERE decision='QUARANTINE'").show(5, truncate=False)
q.stop()

---
### Recap
- The guardrail query never changed — only the **trigger** did.
- `Trigger.RealTime` is the Spark 4.1 Real-Time Mode API; today it's reached via the JVM (no PySpark kwarg yet).
- Real-Time Mode needs a **Kafka source** and runs on **stateless** ops — both true here.